# 怎样在同一预算下比较分类模型？：参考答案

每道题先写预期结果再运行。练习中的 TODO 需要自行完成；能顺序运行不代表已完成练习。

In [1]:
from pathlib import Path
CASE_DIR = Path.cwd()
# 载入本例的可修改函数和数据，所有算法都在同目录 solution.py。
exec(compile((CASE_DIR / "solution.py").read_text(encoding="utf-8"), str(CASE_DIR / "solution.py"), "exec"))

{'selected_before_test': 'logistic', 'folds': 3, 'candidates_per_model': 3, 'comparison': [{'model': 'logistic', 'cv_score': 0.9679527948554872, 'seconds': 0.06509190000360832, 'best_parameters': {'logisticregression__C': 1}, 'test_balanced_accuracy': 0.970020964360587}, {'model': 'boosting', 'cv_score': 0.9504275316232068, 'seconds': 0.8203878999920562, 'best_parameters': {'max_leaf_nodes': 3}, 'test_balanced_accuracy': 0.9683438155136268}]}


## 1. 补全：验证折没有交集
每个折的训练和验证索引应互斥，而且合计覆盖开发集。

In [2]:
for train_ids,val_ids in folds:
    assert set(train_ids).isdisjoint(val_ids)
    assert len(train_ids)+len(val_ids) == len(X_dev)
print('3 折划分检查通过')

3 折划分检查通过


## 2. 修改：查看被选方案的混淆矩阵
在测试集按固定选型结果计算混淆矩阵，并核对样本量。

In [3]:
matrix = confusion_matrix(y_test,result[selected],labels=[0,1])
assert matrix.sum() == len(y_test)
print(matrix)

[[88  2]
 [ 2 51]]


## 3. 迁移：另一份多分类数据
用 sklearn 内置 wine，独立留出 25%，仅比较固定配置的逻辑回归；保证预测类别合法。

In [4]:
from sklearn.datasets import load_wine
wine = load_wine()
wx,tx,wy,ty = train_test_split(wine.data,wine.target,test_size=0.25,stratify=wine.target,random_state=101)
transfer = make_pipeline(StandardScaler(),LogisticRegression(max_iter=500)).fit(wx,wy)
pred = transfer.predict(tx)
assert set(pred).issubset({0,1,2})
print('wine balanced accuracy',balanced_accuracy_score(ty,pred))

wine balanced accuracy 0.9537037037037037
